# Experimento 3 — mTLS + Keycloak


## Celda 1 — Instalar dependencias del sistema

In [13]:
import subprocess, os, stat

def run(cmd, **kw):
    print(f'  $ {cmd if isinstance(cmd, str) else " ".join(cmd)}')
    kw.pop('shell', None)
    result = subprocess.run(
        cmd, shell=isinstance(cmd, str),
        capture_output=True, text=True, **kw
    )
    if result.stdout.strip(): print(result.stdout.strip())
    if result.returncode != 0:
        print(f'  STDERR: {result.stderr.strip()}')
        raise RuntimeError(f'Falló: {cmd}')
    return result.stdout.strip()

def instalar_binario(nombre, url, destino='/usr/local/bin'):
    tmp_tar = f'/tmp/{nombre}.tar.gz'
    tmp_dir = f'/tmp/{nombre}_extract'

    print(f'  Descargando {nombre}...')
    run(f'wget -q "{url}" -O {tmp_tar}')

    os.makedirs(tmp_dir, exist_ok=True)
    run(f'tar -xzf {tmp_tar} -C {tmp_dir}')

    # Busca el binario en cualquier nivel del tar (sin asumir la ruta)
    matches = []
    for root, dirs, files in os.walk(tmp_dir):
        for f in files:
            if f == nombre:
                matches.append(os.path.join(root, f))

    if not matches:
        run(f'find {tmp_dir} -type f')  # muestra qué había dentro para debug
        raise FileNotFoundError(f'No se encontró el binario "{nombre}" en el tar.gz')

    src = matches[0]
    dst = os.path.join(destino, nombre)
    run(f'cp {src} {dst}')
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    print(f'  Instalado en {dst}')

run('apt-get update -qq')

run('apt-get install -y -qq openjdk-17-jdk curl unzip wget')
print(f'  Java: {run("java -version 2>&1 | head -1", shell=True)}')

STEP_VERSION   = '0.25.2'
STEPCA_VERSION = '0.25.2'

instalar_binario(
    'step',
    f'https://github.com/smallstep/cli/releases/download/v{STEP_VERSION}/step_linux_{STEP_VERSION}_amd64.tar.gz'
)
instalar_binario(
    'step-ca',
    f'https://github.com/smallstep/certificates/releases/download/v{STEPCA_VERSION}/step-ca_linux_{STEPCA_VERSION}_amd64.tar.gz'
)
print(f'  step:    {run("step version 2>&1 | head -1", shell=True)}')
print(f'  step-ca: {run("step-ca version 2>&1 | head -1", shell=True)}')

KC_VERSION = '24.0.5'
if not os.path.isdir('/opt/keycloak'):
    run(f'wget -q "https://github.com/keycloak/keycloak/releases/download/{KC_VERSION}/keycloak-{KC_VERSION}.zip" -O /tmp/keycloak.zip')
    run(f'unzip -q /tmp/keycloak.zip -d /opt')
    run(f'mv /opt/keycloak-{KC_VERSION} /opt/keycloak')
    os.chmod('/opt/keycloak/bin/kc.sh', 0o755)
    print('  Keycloak instalado en /opt/keycloak')
else:
    print('  Keycloak ya existe, omitiendo descarga')

run('pip install -q fastapi==0.111.0 "uvicorn[standard]==0.29.0" "python-jose[cryptography]==3.3.0" requests==2.31.0 cryptography==42.0.8')

print()
print('=' * 50)

=== Paso 1/4: Actualizando apt ===
  $ apt-get update -qq

=== Paso 2/4: Instalando Java 17 ===
  $ apt-get install -y -qq openjdk-17-jdk curl unzip wget
  $ java -version 2>&1 | head -1
openjdk version "17.0.18" 2026-01-20
  Java: openjdk version "17.0.18" 2026-01-20

=== Paso 3/4: Instalando step y step-ca ===
  Descargando step...
  $ wget -q "https://github.com/smallstep/cli/releases/download/v0.25.2/step_linux_0.25.2_amd64.tar.gz" -O /tmp/step.tar.gz
  $ tar -xzf /tmp/step.tar.gz -C /tmp/step_extract
  $ cp /tmp/step_extract/step_0.25.2/bin/step /usr/local/bin/step
  Instalado en /usr/local/bin/step
  Descargando step-ca...
  $ wget -q "https://github.com/smallstep/certificates/releases/download/v0.25.2/step-ca_linux_0.25.2_amd64.tar.gz" -O /tmp/step-ca.tar.gz
  $ tar -xzf /tmp/step-ca.tar.gz -C /tmp/step-ca_extract
  $ cp /tmp/step-ca_extract/step-ca /usr/local/bin/step-ca
  Instalado en /usr/local/bin/step-ca
  $ step version 2>&1 | head -1
Smallstep CLI/0.25.2 (linux/amd64)
  s

## Celda 2 — Inicializar la CA y generar certificados

In [14]:
import subprocess, os, time

def run(cmd, env=None):
    e = {**os.environ, **(env or {})}
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=e)
    if result.stdout.strip(): print(result.stdout.strip())
    if result.returncode != 0:
        print(f'STDERR: {result.stderr.strip()}')
        raise RuntimeError(f'Falló: {cmd}')
    return result.stdout.strip()

CA_HOME   = '/root/.step-ca'
CERTS     = '/root/certs'
PASS      = 'CaPassword1234!'
PASS_FILE = '/root/.ca-password'

os.makedirs(CERTS, exist_ok=True)
with open(PASS_FILE, 'w') as f: f.write(PASS)
os.chmod(PASS_FILE, 0o600)

if not os.path.exists(f'{CA_HOME}/config/ca.json'):
    print('=== Inicializando step-ca ===')
    run(f'''STEPPATH={CA_HOME} step ca init \
        --name Exp3-CA \
        --dns localhost \
        --address 127.0.0.1:9000 \
        --provisioner admin \
        --password-file {PASS_FILE} \
        --deployment-type standalone \
        --no-db''')
    print('CA inicializada')
else:
    print('ℹ️  CA ya existe, omitiendo init')

run(f'cp {CA_HOME}/certs/root_ca.crt {CERTS}/ca.crt')
print('ca.crt copiado')

print('\n=== Arrancando step-ca para emitir certs ===')
run('pkill step-ca 2>/dev/null || true')
time.sleep(1)

log = open('/tmp/step-ca-setup.log', 'w')
proc = subprocess.Popen(
    f'STEPPATH={CA_HOME} step-ca --password-file {PASS_FILE} {CA_HOME}/config/ca.json',
    shell=True, stdout=log, stderr=log
)

print('Esperando step-ca', end='')
listo = False
for _ in range(20):
    time.sleep(1)
    print('.', end='', flush=True)
    r = subprocess.run('curl -sk https://localhost:9000/health',
                       shell=True, capture_output=True, text=True)
    if 'ok' in r.stdout:
        listo = True
        break

print()
if not listo:
    print(open('/tmp/step-ca-setup.log').read())
    raise RuntimeError('step-ca no respondió')

def emitir(cn, fname):
    print(f'  → Emitiendo cert para: {cn}')
    run(f'''STEPPATH={CA_HOME} step ca certificate {cn} \
        {CERTS}/{fname}.crt {CERTS}/{fname}.key \
        --ca-url https://localhost:9000 \
        --root {CERTS}/ca.crt \
        --provisioner admin \
        --password-file {PASS_FILE} \
        --not-after 24h \
        --force''')

print('\n=== Emitiendo certificados ===')
emitir('server',         'server')
emitir('service-client', 'client')

proc.terminate()

print('\n Certificados generados:')
run(f'ls -lh {CERTS}/')

ℹ️  CA ya existe, omitiendo init
ca.crt copiado

=== Arrancando step-ca para emitir certs ===
Esperando step-ca.

=== Emitiendo certificados ===
  → Emitiendo cert para: server
  → Emitiendo cert para: service-client

 Certificados generados:
total 20K
-rw------- 1 root root  615 Mar 27 16:21 ca.crt
-rw------- 1 root root 1.5K Mar 27 16:21 client.crt
-rw------- 1 root root  227 Mar 27 16:21 client.key
-rw------- 1 root root 1.5K Mar 27 16:21 server.crt
-rw------- 1 root root  227 Mar 27 16:21 server.key


'total 20K\n-rw------- 1 root root  615 Mar 27 16:21 ca.crt\n-rw------- 1 root root 1.5K Mar 27 16:21 client.crt\n-rw------- 1 root root  227 Mar 27 16:21 client.key\n-rw------- 1 root root 1.5K Mar 27 16:21 server.crt\n-rw------- 1 root root  227 Mar 27 16:21 server.key'

## Celda 3 — Escribir el servidor FastAPI

In [16]:
server_code = '''
import ssl, os, logging
import requests
import uvicorn
from fastapi import FastAPI, Request, HTTPException, Depends
from fastapi.security import HTTPBearer, HTTPAuthorizationCredentials
from jose import jwt, JWTError

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

KEYCLOAK_URL   = os.getenv("KEYCLOAK_URL",   "http://localhost:8080")
KEYCLOAK_REALM = os.getenv("KEYCLOAK_REALM", "exp3")
CERT_DIR       = os.getenv("CERT_DIR",       "/root/certs")
JWKS_URL       = f"{KEYCLOAK_URL}/realms/{KEYCLOAK_REALM}/protocol/openid-connect/certs"

app  = FastAPI(title="Experimento 3 — mTLS + Keycloak")
auth = HTTPBearer(auto_error=False)

@app.get("/health")
async def health():
    return {"status": "ok"}

@app.get("/servicio/datos", tags=["mTLS — identidad de maquina"])
async def datos_servicio():
    """Solo mTLS — si llegaste aqui, tu certificado fue validado por TLS."""
    return {
        "mensaje":       "Acceso concedido — identidad de MAQUINA verificada",
        "autenticacion": "mTLS (certificado de step-ca)",
        "concepto":      "El servicio que llama tiene identidad criptografica fuerte",
    }

@app.get("/usuario/datos", tags=["JWT — identidad de usuario"])
async def datos_usuario(credentials: HTTPAuthorizationCredentials = Depends(auth)):
    """mTLS + JWT — identidad de usuario via Keycloak."""
    if not credentials:
        raise HTTPException(401, "Header: Authorization: Bearer <token>")
    payload = _verificar_jwt(credentials.credentials)
    return {
        "mensaje":       "Acceso concedido — identidad de USUARIO verificada",
        "autenticacion": "JWT (Keycloak)",
        "usuario":       payload.get("preferred_username", "—"),
        "roles":         payload.get("realm_access", {}).get("roles", []),
    }

@app.get("/completo", tags=["mTLS + JWT — el punto del Exp 3"])
async def completo(credentials: HTTPAuthorizationCredentials = Depends(auth)):
    """Demuestra la diferencia: MAQUINA (cert) + USUARIO (JWT)."""
    if not credentials:
        raise HTTPException(401, "Se requiere JWT ademas del certificado")
    payload = _verificar_jwt(credentials.credentials)
    return {
        "mensaje":           "Ambas identidades verificadas",
        "identidad_maquina": "certificado step-ca (mTLS)",
        "identidad_usuario": payload.get("preferred_username", "—"),
        "roles":             payload.get("realm_access", {}).get("roles", []),
        "conclusion":        "Un SERVICIO con cert actua en nombre de un USUARIO de Keycloak",
    }

def _verificar_jwt(token):
    try:
        jwks = requests.get(JWKS_URL, timeout=5).json()
    except Exception:
        raise HTTPException(503, f"No se pudo contactar Keycloak: {JWKS_URL}")
    try:
        return jwt.decode(token, jwks, algorithms=["RS256"], options={"verify_aud": False})
    except JWTError as e:
        raise HTTPException(401, f"Token invalido: {e}")

if __name__ == "__main__":
    for f in [f"{CERT_DIR}/server.crt", f"{CERT_DIR}/server.key", f"{CERT_DIR}/ca.crt"]:
        if not os.path.exists(f):
            logger.error(f"No encontrado: {f}")
            raise SystemExit(1)
    logger.info("Servidor mTLS arrancando en :8443")
    uvicorn.run(app, host="0.0.0.0", port=8443,
                ssl_certfile=f"{CERT_DIR}/server.crt",
                ssl_keyfile=f"{CERT_DIR}/server.key",
                ssl_ca_certs=f"{CERT_DIR}/ca.crt",
                ssl_cert_reqs=ssl.CERT_REQUIRED,
                log_level="warning")
'''

with open('/root/server.py', 'w') as f:
    f.write(server_code)

print('server.py escrito en /root/server.py')
print('Siguiente: Celda 4')

server.py escrito en /root/server.py
Siguiente: Celda 4


## Celda 4 — Arrancar los 3 servicios

In [17]:
import subprocess, os, time

r = subprocess.run('curl -sf http://localhost:8180/health/ready', shell=True, capture_output=True, text=True)
print("Keycloak:", r.stdout or " no responde (puede que tarde más en arrancar)")

os.makedirs('/root/logs', exist_ok=True)

log = open('/root/logs/server.log', 'w')
subprocess.Popen(
    'python3 /root/server.py',
    shell=True, stdout=log, stderr=log,
    env={**os.environ, 'CERT_DIR': '/root/certs',
         'KEYCLOAK_URL': 'http://localhost:8180',
         'KEYCLOAK_REALM': 'exp3'}
)

time.sleep(4)
r = subprocess.run(
    'curl -sk --cert /root/certs/client.crt --key /root/certs/client.key '
    '--cacert /root/certs/ca.crt https://localhost:8443/health',
    shell=True, capture_output=True, text=True)
print("FastAPI:", r.stdout or "no responde")
print(open('/root/logs/server.log').read()[-300:])

Keycloak:  no responde (puede que tarde más en arrancar)
FastAPI: {"status":"ok"}
2026-03-27 16:21:52,114 [INFO] Servidor mTLS arrancando en :8443
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8443): address already in use



In [18]:
import subprocess, os, time

KC_HOME = '/opt/keycloak'

subprocess.run('pkill -f "keycloak" 2>/dev/null', shell=True)
time.sleep(2)

# Arrancar Keycloak en modo dev (sin TLS, puerto 8180)
os.makedirs('/root/logs', exist_ok=True)
kc_log = open('/root/logs/keycloak.log', 'w')
subprocess.Popen(
    f'{KC_HOME}/bin/kc.sh start-dev --http-port=8180',
    shell=True, stdout=kc_log, stderr=kc_log,
    env={**os.environ,
         'KEYCLOAK_ADMIN': 'admin',
         'KEYCLOAK_ADMIN_PASSWORD': 'admin',
         'KC_HTTP_ENABLED': 'true',
         'KC_HOSTNAME_STRICT': 'false',
         'JAVA_HOME': '/usr/lib/jvm/java-17-openjdk-amd64'}
)

# Esperar a que Keycloak esté listo (puede tardar 60-90 segundos)
print('Esperando Keycloak', end='')
listo = False
for _ in range(60):
    time.sleep(3)
    print('.', end='', flush=True)
    r = subprocess.run(
        'curl -sf http://localhost:8180/health/ready',
        shell=True, capture_output=True, text=True)
    if 'UP' in r.stdout or '"status":"UP"' in r.stdout:
        listo = True
        break

print()
if listo:
    print('Keycloak listo en http://localhost:8180')
else:
    print('Keycloak no respondió. Últimas líneas del log:')
    print(open('/root/logs/keycloak.log').read()[-1000:])

Esperando Keycloak............................................................
Keycloak no respondió. Últimas líneas del log:
JBossUserMarshaller'
2026-03-27 16:22:15,928 INFO  [org.keycloak.broker.provider.AbstractIdentityProviderMapper] (main) Registering class org.keycloak.broker.provider.mappersync.ConfigSyncEventListener
2026-03-27 16:22:15,966 INFO  [org.keycloak.connections.infinispan.DefaultInfinispanConnectionProviderFactory] (main) Node name: node_128183, Site name: null
2026-03-27 16:22:18,006 INFO  [io.quarkus] (main) Keycloak 24.0.5 on JVM (powered by Quarkus 3.8.4) started in 11.120s. Listening on: http://0.0.0.0:8180
2026-03-27 16:22:18,007 INFO  [io.quarkus] (main) Profile dev activated. 
2026-03-27 16:22:18,007 INFO  [io.quarkus] (main) Installed features: [agroal, cdi, hibernate-orm, jdbc-h2, keycloak, logging-gelf, narayana-jta, reactive-routes, resteasy-reactive, resteasy-reactive-jackson, smallrye-context-propagation, vertx]
2026-03-27 16:22:18,012 WARN  [org.keycl

## Celda 5 — Configurar Keycloak (realm + usuarios)

In [20]:
import requests

KC = 'http://localhost:8180'
REALM = 'exp3'

# Token admin
token = requests.post(f'{KC}/realms/master/protocol/openid-connect/token',
    data={'grant_type':'password','client_id':'admin-cli',
          'username':'admin','password':'admin'}).json()['access_token']
H = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}
print('Token admin obtenido')

# Crear realm
realms = [x['realm'] for x in requests.get(f'{KC}/admin/realms', headers=H).json()]
if REALM not in realms:
    requests.post(f'{KC}/admin/realms', headers=H,
        json={'realm': REALM, 'enabled': True, 'accessTokenLifespan': 3600})
    print(f'Realm "{REALM}" creado')
else:
    print(f'Realm "{REALM}" ya existe')

# Habilitar Direct Access Grants en admin-cli del realm exp3
clients = requests.get(f'{KC}/admin/realms/{REALM}/clients', headers=H).json()
admin_cli = next((c for c in clients if c['clientId'] == 'admin-cli'), None)
if admin_cli:
    admin_cli['directAccessGrantsEnabled'] = True
    requests.put(f'{KC}/admin/realms/{REALM}/clients/{admin_cli["id"]}',
        headers=H, json=admin_cli)
    print('Direct Access Grants habilitado en admin-cli')

# Crear roles
for rol in ['admin-role', 'user-role']:
    roles = [r['name'] for r in requests.get(f'{KC}/admin/realms/{REALM}/roles', headers=H).json()]
    if rol not in roles:
        requests.post(f'{KC}/admin/realms/{REALM}/roles', headers=H, json={'name': rol})
        print(f'Rol "{rol}" creado')
    else:
        print(f'ℹ  Rol "{rol}" ya existe')

# Crear usuarios
def crear_usuario(username, password, email, rol):
    existentes = requests.get(f'{KC}/admin/realms/{REALM}/users?username={username}', headers=H).json()
    if existentes:
        print(f'Usuario "{username}" ya existe')
        return
    requests.post(f'{KC}/admin/realms/{REALM}/users', headers=H, json={
        'username': username, 'email': email,
        'enabled': True, 'emailVerified': True,
        'credentials': [{'type':'password','value':password,'temporary':False}]
    })
    uid      = requests.get(f'{KC}/admin/realms/{REALM}/users?username={username}', headers=H).json()[0]['id']
    role_obj = requests.get(f'{KC}/admin/realms/{REALM}/roles/{rol}', headers=H).json()
    requests.post(f'{KC}/admin/realms/{REALM}/users/{uid}/role-mappings/realm',
        headers=H, json=[{'id': role_obj['id'], 'name': rol}])
    print(f'Usuario "{username}" creado con rol "{rol}"')

crear_usuario('admin-user', 'Admin1234!', 'admin@exp3.local', 'admin-role')
crear_usuario('test-user',  'User1234!',  'user@exp3.local',  'user-role')

# Verificar que el login funciona
print('\n=== Verificando JWT de test-user ===')
r = requests.post(f'{KC}/realms/{REALM}/protocol/openid-connect/token',
    data={'grant_type':'password','client_id':'admin-cli',
          'username':'test-user','password':'User1234!'})
print('Status:', r.status_code)
if r.status_code == 200:
    print('JWT de test-user OK')
else:
    print(r.text[:200])

print('\nKeycloak configurado — corre la Celda 6 de nuevo')

Token admin obtenido
Realm "exp3" ya existe
Direct Access Grants habilitado en admin-cli
ℹ️  Rol "admin-role" ya existe
ℹ️  Rol "user-role" ya existe
ℹ️  Usuario "admin-user" ya existe
ℹ️  Usuario "test-user" ya existe

=== Verificando JWT de test-user ===
Status: 200
JWT de test-user OK

Keycloak configurado — corre la Celda 6 de nuevo


In [22]:
import requests

KC = 'http://localhost:8180'
REALM = 'exp3'

token = requests.post(f'{KC}/realms/master/protocol/openid-connect/token',
    data={'grant_type':'password','client_id':'admin-cli',
          'username':'admin','password':'admin'}).json()['access_token']
H = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

# Deshabilitar VERIFY_PROFILE en el realm
r = requests.get(f'{KC}/admin/realms/{REALM}/authentication/required-actions/VERIFY_PROFILE', headers=H)
if r.status_code == 200:
    action = r.json()
    action['enabled'] = False
    action['defaultAction'] = False
    requests.put(f'{KC}/admin/realms/{REALM}/authentication/required-actions/VERIFY_PROFILE', headers=H, json=action)
    print('VERIFY_PROFILE deshabilitado')

# Borrar usuarios existentes
for username in ['test-user', 'admin-user']:
    users = requests.get(f'{KC}/admin/realms/{REALM}/users?username={username}', headers=H).json()
    if users:
        requests.delete(f'{KC}/admin/realms/{REALM}/users/{users[0]["id"]}', headers=H)
        print(f' {username} eliminado')

# Recrear usuarios limpios
def crear_usuario(username, password, email, rol):
    r = requests.post(f'{KC}/admin/realms/{REALM}/users', headers=H, json={
        'username': username,
        'email': email,
        'firstName': username,
        'lastName': 'Exp3',
        'enabled': True,
        'emailVerified': True,
        'requiredActions': [],
        'credentials': [{'type':'password','value':password,'temporary':False}]
    })
    print(f'  Crear usuario status: {r.status_code}')
    uid = requests.get(f'{KC}/admin/realms/{REALM}/users?username={username}', headers=H).json()[0]['id']
    role_obj = requests.get(f'{KC}/admin/realms/{REALM}/roles/{rol}', headers=H).json()
    requests.post(f'{KC}/admin/realms/{REALM}/users/{uid}/role-mappings/realm',
        headers=H, json=[{'id': role_obj['id'], 'name': rol}])
    print(f'{username} creado con rol {rol}')

crear_usuario('test-user',  'User1234!',  'user@exp3.local',  'user-role')
crear_usuario('admin-user', 'Admin1234!', 'admin@exp3.local', 'admin-role')

# Verificar
print('\n=== Verificando logins ===')
for username, password in [('test-user', 'User1234!'), ('admin-user', 'Admin1234!')]:
    r = requests.post(f'{KC}/realms/{REALM}/protocol/openid-connect/token',
        data={'grant_type':'password','client_id':'admin-cli',
              'username': username, 'password': password})
    if r.status_code == 200:
        print(f'{username} — JWT OK')
    else:
        print(f'{username} — {r.json().get("error_description")}')

VERIFY_PROFILE deshabilitado
 test-user eliminado
 admin-user eliminado
  Crear usuario status: 201
test-user creado con rol user-role
  Crear usuario status: 201
admin-user creado con rol admin-role

=== Verificando logins ===
test-user — JWT OK
admin-user — JWT OK


## Celda 6 — Tests completos del Experimento 3

In [23]:
import requests, json, warnings, os, tempfile, datetime
warnings.filterwarnings('ignore')

from cryptography import x509
from cryptography.x509.oid import NameOID
from cryptography.hazmat.primitives import hashes, serialization
from cryptography.hazmat.primitives.asymmetric import rsa

BASE        = 'https://localhost:8443'
KC          = 'http://localhost:8180'
REALM       = 'exp3'
CERTS       = '/root/certs'
CLIENT_CERT = (f'{CERTS}/client.crt', f'{CERTS}/client.key')

G='\033[92m'; R='\033[91m'; Y='\033[93m'; B='\033[94m'; BOLD='\033[1m'; X='\033[0m'
ok   = lambda m: print(f'  {G} {m}{X}')
fail = lambda m: print(f'  {R} {m}{X}')
info = lambda m: print(f'  {B}  {m}{X}')
sep  = lambda t: print(f'\n{BOLD}{B}{"─"*52}\n  {t}\n{"─"*52}{X}')

def call(endpoint, con_cert=True, token=None):
    h = {'Authorization': f'Bearer {token}'} if token else {}
    return requests.get(f'{BASE}{endpoint}',
        cert=CLIENT_CERT if con_cert else None,
        verify=False, headers=h, timeout=5)

def get_jwt(username, password):
    r = requests.post(f'{KC}/realms/{REALM}/protocol/openid-connect/token',
        data={'grant_type':'password','client_id':'admin-cli',
              'username': username, 'password': password}, timeout=5)
    return r.json().get('access_token') if r.status_code == 200 else None

resultados = {}

# TEST 1
sep('TEST 1 — Sin certificado → debe fallar')
try:
    call('/servicio/datos', con_cert=False)
    fail('INESPERADO: aceptó sin certificado'); resultados['T1'] = False
except Exception as e:
    if 'RemoteDisconnected' in str(e) or 'SSLError' in str(e) or 'Connection aborted' in str(e):
        ok('Conexión rechazada por el servidor — sin cert no hay acceso')
        ok('El rechazo ocurre a nivel TLS, antes que FastAPI lo vea')
        resultados['T1'] = True
    else:
        fail(f'Error inesperado: {e}'); resultados['T1'] = False

# TEST 2
sep('TEST 2 — Con certificado válido → debe funcionar (200 OK)')
try:
    r = call('/servicio/datos')
    if r.status_code == 200:
        ok('Status 200 — identidad de MÁQUINA verificada')
        print(json.dumps(r.json(), indent=4, ensure_ascii=False))
        resultados['T2'] = True
    else:
        fail(f'Status: {r.status_code}'); resultados['T2'] = False
except Exception as e:
    fail(str(e)); resultados['T2'] = False

# TEST 3
sep('TEST 3 — Certificado falso → debe fallar')
key = rsa.generate_private_key(public_exponent=65537, key_size=2048)
cert = (x509.CertificateBuilder()
    .subject_name(x509.Name([x509.NameAttribute(NameOID.COMMON_NAME,'hacker')]))
    .issuer_name(x509.Name([x509.NameAttribute(NameOID.COMMON_NAME,'fake-ca')]))
    .public_key(key.public_key())
    .serial_number(x509.random_serial_number())
    .not_valid_before(datetime.datetime.utcnow())
    .not_valid_after(datetime.datetime.utcnow() + datetime.timedelta(days=1))
    .sign(key, hashes.SHA256()))

with tempfile.NamedTemporaryFile(suffix='.crt', delete=False) as f:
    fake_crt=f.name; f.write(cert.public_bytes(serialization.Encoding.PEM))
with tempfile.NamedTemporaryFile(suffix='.key', delete=False) as f:
    fake_key=f.name; f.write(key.private_bytes(
        serialization.Encoding.PEM,
        serialization.PrivateFormat.TraditionalOpenSSL,
        serialization.NoEncryption()))
try:
    requests.get(f'{BASE}/servicio/datos', cert=(fake_crt,fake_key), verify=False, timeout=5)
    fail('INESPERADO: aceptó cert falso'); resultados['T3'] = False
except Exception as e:
    if 'RemoteDisconnected' in str(e) or 'SSLError' in str(e) or 'Connection aborted' in str(e):
        ok('Conexión rechazada — cert falso no aceptado')
        ok('Solo certs emitidos por step-ca son válidos')
        resultados['T3'] = True
    else:
        fail(f'Error inesperado: {e}'); resultados['T3'] = False
finally:
    os.unlink(fake_crt); os.unlink(fake_key)

# TEST 4
sep('TEST 4 — Identidad de usuario con Keycloak (mTLS + JWT)')
info('Paso 1: login en Keycloak como test-user...')
token = get_jwt('test-user', 'User1234!')
if not token:
    fail('No se pudo obtener JWT'); resultados['T4'] = False
else:
    ok(f'JWT obtenido ({token[:45]}...)')
    info('Paso 2: llamar /usuario/datos con cert + JWT...')
    try:
        r = call('/usuario/datos', token=token)
        if r.status_code == 200:
            d = r.json()
            ok(f"Usuario: {d.get('usuario')}  |  Roles: {d.get('roles')}")
            print(json.dumps(d, indent=4, ensure_ascii=False))
            resultados['T4'] = True
        else:
            fail(f'Status: {r.status_code} — {r.text[:200]}')
            resultados['T4'] = False
    except Exception as e:
        fail(str(e)); resultados['T4'] = False

# TEST 5
sep('TEST 5 — El punto del Exp 3: máquina + usuario en /completo')
info('Identidad de MÁQUINA  →  certificado step-ca (mTLS)')
info('Identidad de USUARIO  →  JWT de Keycloak')
token = get_jwt('admin-user', 'Admin1234!')
if not token:
    fail('No se pudo obtener JWT'); resultados['T5'] = False
else:
    try:
        r = call('/completo', token=token)
        if r.status_code == 200:
            d = r.json()
            ok('Ambas identidades verificadas')
            print(json.dumps(d, indent=4, ensure_ascii=False))
            ok(f"Máquina  → {d.get('identidad_maquina')}")
            ok(f"Usuario  → {d.get('identidad_usuario')}  {d.get('roles')}")
            resultados['T5'] = True
        else:
            fail(f'Status: {r.status_code}'); resultados['T5'] = False
    except Exception as e:
        fail(str(e)); resultados['T5'] = False

# Resumen
sep('RESUMEN DE RESULTADOS')
nombres = {
    'T1': 'Sin certificado (debe fallar)',
    'T2': 'Con certificado válido',
    'T3': 'Certificado falso (debe fallar)',
    'T4': 'JWT Keycloak (identidad de usuario)',
    'T5': 'Autenticación combinada (máquina + usuario)',
}
for k,v in resultados.items():
    (ok if v else fail)(nombres[k])

print()
if all(resultados.values()):
    print(f'{G}{BOLD} Experimento 3 completado{X}')
else:
    print(f'{Y}Algunos tests fallaron.{X}')


────────────────────────────────────────────────────
  TEST 1 — Sin certificado → debe fallar
────────────────────────────────────────────────────
   Conexión rechazada por el servidor — sin cert no hay acceso
   El rechazo ocurre a nivel TLS, antes que FastAPI lo vea

────────────────────────────────────────────────────
  TEST 2 — Con certificado válido → debe funcionar (200 OK)
────────────────────────────────────────────────────
   Status 200 — identidad de MÁQUINA verificada
{
    "mensaje": "Acceso concedido — identidad de MAQUINA verificada",
    "autenticacion": "mTLS (certificado de step-ca)",
    "concepto": "El servicio que llama tiene identidad criptografica fuerte"
}

────────────────────────────────────────────────────
  TEST 3 — Certificado falso → debe fallar
────────────────────────────────────────────────────
   Conexión rechazada — cert falso no aceptado
   Solo certs emitidos por step-ca son válidos

────────────────────────────────────────────────────
  TEST 4 — Id

In [12]:
import subprocess, os

# Matar procesos anteriores
subprocess.run('pkill -f step-ca; pkill -f keycloak; pkill -f uvicorn', shell=True)

# Forzar reinstalación
for f in ['/usr/local/bin/step', '/usr/local/bin/step-ca']:
    try:
        os.remove(f)
        print(f'Eliminado: {f}')
    except FileNotFoundError:
        print(f'No existía: {f}')

print('Listo — ahora corre la Celda 1 de nuevo')

Eliminado: /usr/local/bin/step
Eliminado: /usr/local/bin/step-ca
Listo — ahora corre la Celda 1 de nuevo
